In [19]:
import pandas as pd

# 1. Load your original file
df = pd.read_excel("C:/Users/Ex0164/series_part_unmapped.xlsx", sheet_name="Sheet1")

# Standardize column names (in case Excel added extra spaces or different naming)
df.columns = ['label', 'count']

# Remove useless rows
df = df[df['label'].notna()]
df = df[~df['label'].isin(['Row Labels', 'Grand Total', '(blank)'])]
df['label'] = df['label'].astype(str).str.strip()
df['count'] = pd.to_numeric(df['count'], errors='coerce').fillna(0).astype(int)

# 2. Build flat mapping table
output_rows = []
i = 0
current_main = None
current_main_count = 0
sub_sum = 0

while i < len(df):
    label = df.iloc[i]['label']
    cnt = df.iloc[i]['count']
    
    # Heuristic: new main when count matches expected pattern or after subs complete
    if current_main is None or sub_sum >= current_main_count:
        current_main = label
        current_main_count = cnt
        sub_sum = 0
        # Add main row (optional - you can skip if you only want subs)
        output_rows.append({
            'Main_Label': current_main,
            'Main_Count': current_main_count,
            'Sub_Label': '',
            'Sub_Count': ''
        })
    else:
        # This is a sub-item
        output_rows.append({
            'Main_Label': current_main,
            'Main_Count': current_main_count,
            'Sub_Label': label,
            'Sub_Count': cnt
        })
        sub_sum += cnt
    
    i += 1

# 3. Create DataFrame and save
result_df = pd.DataFrame(output_rows)

# Optional: fill down main values for better readability in Excel
result_df['Main_Label'] = result_df['Main_Label'].replace('', pd.NA).ffill()
result_df['Main_Count'] = result_df['Main_Count'].replace('', pd.NA).ffill()

result_df.to_excel("main_sub_mapping.xlsx", index=False)
print(f"Saved {len(result_df)} rows to main_sub_mapping.xlsx")
print("\nFirst 10 rows preview:")
print(result_df.head(10))

Saved 136129 rows to main_sub_mapping.xlsx

First 10 rows preview:
           Main_Label  Main_Count           Sub_Label Sub_Count
0  14CL510026-00001X0           2                              
1  14CL510026-00001X0           2   14CL510026-0004E0         1
2  14CL510026-00001X0           2  14CL510026-00701S0         1
3  14CL510026-00001X1           2                              
4  14CL510026-00001X1           2   14CL510026-0004E0         1
5  14CL510026-00001X1           2  14CL510026-00701S0         1
6  14CL510026-00002X0           2                              
7  14CL510026-00002X0           2   14CL510026-0004E0         1
8  14CL510026-00002X0           2  14CL510026-00701S0         1
9  14CL510026-00003Y0           2                              


In [20]:
import pandas as pd

# ────────────────────────────────────────────────
# 1. Load your file
# ────────────────────────────────────────────────
df = pd.read_excel("C:/Users/Ex0164/series_part_unmapped.xlsx", sheet_name="Sheet1")

# Standardize columns
df.columns = ['label', 'count'] if len(df.columns) == 2 else df.columns[:2]
df = df[df['label'].notna()]
df = df[~df['label'].isin(['Row Labels', 'Grand Total', '(blank)'])]
df['label'] = df['label'].astype(str).str.strip()
df['count'] = pd.to_numeric(df['count'], errors='coerce').fillna(0).astype(int)

# ────────────────────────────────────────────────
# 2. Parse hierarchy into flat table
# ────────────────────────────────────────────────
output_rows = []
i = 0
current_main = None
current_main_count = 0
sub_sum = 0

while i < len(df):
    label = df.iloc[i]['label']
    cnt = df.iloc[i]['count']
    
    # New main block starts
    if current_main is None or sub_sum >= current_main_count:
        current_main = label
        current_main_count = cnt
        sub_sum = 0
        # Add main header row
        output_rows.append({
            'Main_Label': current_main,
            'Main_Count': current_main_count,
            'Sub_Label': '',
            'Sub_Count': ''
        })
    else:
        # Sub item
        output_rows.append({
            'Main_Label': current_main,
            'Main_Count': current_main_count,
            'Sub_Label': label,
            'Sub_Count': cnt
        })
        sub_sum += cnt
    
    i += 1

result_df = pd.DataFrame(output_rows)

# Fill down main values (for readability)
result_df['Main_Label'] = result_df['Main_Label'].replace('', pd.NA).ffill()
result_df['Main_Count'] = result_df['Main_Count'].replace('', pd.NA).ffill()

# ────────────────────────────────────────────────
# 3. Identify mains that actually have subs
# ────────────────────────────────────────────────
# Group by main and see which have at least one real sub row
has_real_subs = result_df[result_df['Sub_Label'] != ''].groupby('Main_Label').size() > 0
mains_to_keep = has_real_subs[has_real_subs].index

# ────────────────────────────────────────────────
# 4. Filter: keep all sub rows + only mains that have subs
# ────────────────────────────────────────────────
mask = (
    (result_df['Sub_Label'] != '') |                  # all sub rows
    (result_df['Main_Label'].isin(mains_to_keep) & (result_df['Sub_Label'] == ''))  # only good mains
)

clean_df = result_df[mask].copy()

# Optional: reset index, make it look neat
clean_df = clean_df.reset_index(drop=True)

# ────────────────────────────────────────────────
# 5. Save
# ────────────────────────────────────────────────
clean_df.to_excel("main_sub_mapping_clean.xlsx", index=False)

print(f"Original rows in mapping: {len(result_df)}")
print(f"After removing orphan mains: {len(clean_df)}")
print(f"Saved to: main_sub_mapping_clean.xlsx")
print("\nPreview (first 15 rows):")
print(clean_df.head(15))

Original rows in mapping: 136129
After removing orphan mains: 136129
Saved to: main_sub_mapping_clean.xlsx

Preview (first 15 rows):
            Main_Label  Main_Count           Sub_Label Sub_Count
0   14CL510026-00001X0           2                              
1   14CL510026-00001X0           2   14CL510026-0004E0         1
2   14CL510026-00001X0           2  14CL510026-00701S0         1
3   14CL510026-00001X1           2                              
4   14CL510026-00001X1           2   14CL510026-0004E0         1
5   14CL510026-00001X1           2  14CL510026-00701S0         1
6   14CL510026-00002X0           2                              
7   14CL510026-00002X0           2   14CL510026-0004E0         1
8   14CL510026-00002X0           2  14CL510026-00701S0         1
9   14CL510026-00003Y0           2                              
10  14CL510026-00003Y0           2   14CL510026-0004E0         1
11  14CL510026-00003Y0           2  14CL510026-00702S0         1
12  14CL510026-00004X0

In [21]:
import pandas as pd

# 1. Load the original file
df = pd.read_excel("C:/Users/Ex0164/series_part_unmapped.xlsx", sheet_name="Sheet1")

# Standardize columns (in case names differ slightly)
df = df.iloc[:, :2]  # take first two columns
df.columns = ['label', 'count']
df = df[df['label'].notna()]
df = df[~df['label'].isin(['Row Labels', 'Grand Total', '(blank)'])]
df['label'] = df['label'].astype(str).str.strip()
df['count'] = pd.to_numeric(df['count'], errors='coerce').fillna(0).astype(int)

# 2. Build the flat mapping with filled-down mains
output_rows = []
i = 0
current_main = None
current_main_count = 0
sub_sum = 0

while i < len(df):
    label = df.iloc[i]['label']
    cnt = df.iloc[i]['count']
    
    # Detect new main
    if current_main is None or sub_sum >= current_main_count:
        current_main = label
        current_main_count = cnt
        sub_sum = 0
    else:
        # This is a sub → add full row
        output_rows.append({
            'Main_Label': current_main,
            'Main_Count': current_main_count,
            'Sub_Label': label,
            'Sub_Count': cnt
        })
        sub_sum += cnt
    
    i += 1

# 3. Create DataFrame — already only contains sub rows (no empty sub_label rows added)
result_df = pd.DataFrame(output_rows)

# 4. Save — only rows with real subs
result_df.to_excel("main_with_subs_only.xlsx", index=False)

print(f"Rows saved: {len(result_df)} (only sub-rows)")
print("Columns: Main_Label, Main_Count, Sub_Label, Sub_Count")
print("\nFirst 10 rows preview:")
print(result_df.head(10))

Rows saved: 118558 (only sub-rows)
Columns: Main_Label, Main_Count, Sub_Label, Sub_Count

First 10 rows preview:
           Main_Label  Main_Count           Sub_Label  Sub_Count
0  14CL510026-00001X0           2   14CL510026-0004E0          1
1  14CL510026-00001X0           2  14CL510026-00701S0          1
2  14CL510026-00001X1           2   14CL510026-0004E0          1
3  14CL510026-00001X1           2  14CL510026-00701S0          1
4  14CL510026-00002X0           2   14CL510026-0004E0          1
5  14CL510026-00002X0           2  14CL510026-00701S0          1
6  14CL510026-00003Y0           2   14CL510026-0004E0          1
7  14CL510026-00003Y0           2  14CL510026-00702S0          1
8  14CL510026-00004X0           2   14CL510026-0004E0          1
9  14CL510026-00004X0           2  14CL510026-00702S0          1
